# Post-Anomaly Suffix Viewer

Notebook này giúp xem 4 bảng `csv` đã export cho UCR, SMD, SWaT và IOPS theo cách dễ nhìn hơn.

Ý chính ở đây là:
- sort theo `post_anomaly_suffix_len` tăng dần
- tô màu các dòng `suffix = 0`
- tô màu các dòng có `notes = no_anomaly_in_series`
- tô màu các dòng có suffix rất ngắn để dễ rà bằng mắt


In [5]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / ".." / "documents").exists():
    NOTEBOOK_DIR = Path.cwd() / "notebooks"

PROJECT_ROOT = NOTEBOOK_DIR.resolve().parent
REPORT_DIR = PROJECT_ROOT / "documents" / "logs" / "06-27-2026" / "research"

CSV_PATHS = {
    "UCR": REPORT_DIR / "ucr_post_anomaly_suffix_counts.csv",
    "SMD": REPORT_DIR / "smd_post_anomaly_suffix_counts.csv",
    "SWaT": REPORT_DIR / "swat_post_anomaly_suffix_counts.csv",
    "IOPS": REPORT_DIR / "iops_post_anomaly_suffix_counts.csv",
}

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 120)

CSV_PATHS


{'UCR': PosixPath('/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/documents/logs/06-27-2026/research/ucr_post_anomaly_suffix_counts.csv'),
 'SMD': PosixPath('/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/documents/logs/06-27-2026/research/smd_post_anomaly_suffix_counts.csv'),
 'SWaT': PosixPath('/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/documents/logs/06-27-2026/research/swat_post_anomaly_suffix_counts.csv'),
 'IOPS': PosixPath('/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/documents/logs/06-27-2026/research/iops_post_anoma

In [6]:
def load_suffix_table(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    return df.sort_values(
        by=["post_anomaly_suffix_len", "series_id"],
        ascending=[True, True],
        na_position="last",
    ).reset_index(drop=True)


def summarize_suffix_table(df: pd.DataFrame) -> pd.DataFrame:
    suffix = df["post_anomaly_suffix_len"]
    return pd.DataFrame(
        {
            "num_series": [len(df)],
            "min_suffix": [suffix.min()],
            "median_suffix": [suffix.median()],
            "max_suffix": [suffix.max()],
            "zero_suffix_count": [(suffix == 0).sum()],
            "short_suffix_count_le_100": [(suffix <= 100).sum()],
            "no_anomaly_count": [(df["notes"] == "no_anomaly_in_series").sum()],
        }
    )


def style_suffix_table(df: pd.DataFrame):
    def highlight_row(row: pd.Series):
        suffix = row["post_anomaly_suffix_len"]
        notes = row["notes"]
        if notes == "no_anomaly_in_series":
            return ["background-color: #e3f2fd"] * len(row)
        if suffix == 0:
            return ["background-color: #ffebee"] * len(row)
        if suffix <= 100:
            return ["background-color: #fff8e1"] * len(row)
        return [""] * len(row)

    return (
        df.style
        .apply(highlight_row, axis=1)
        .format({"median_suffix": "{:.1f}"}, na_rep="")
    )


In [7]:
tables = {dataset_name: load_suffix_table(csv_path) for dataset_name, csv_path in CSV_PATHS.items()}

for dataset_name, df in tables.items():
    display(Markdown(f"## {dataset_name}"))
    display(summarize_suffix_table(df))
    display(style_suffix_table(df))


## UCR

,num_series,min_suffix,median_suffix,max_suffix,zero_suffix_count,short_suffix_count_le_100,no_anomaly_count
0,250,427,11800.0,304153,0,0,0


,series_id,source_file,total_length,annotation_type,anomaly_end_index,last_anomalous_index,post_anomaly_suffix_len,num_anomalous_points,notes
0,Italianpowerdemand,/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/data/AnomalyArchive/212_UCR_Anomaly_Italianpowerdemand_8913_29480_29504.txt,29931,filename_anomaly_interval,29504,,427,24,anomaly_end_index_from_filename
1,DISTORTEDWalkingAceleration5,/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/data/AnomalyArchive/054_UCR_Anomaly_DISTORTEDWalkingAceleration5_2700_5920_5979.txt,6684,filename_anomaly_interval,5979,,705,59,anomaly_end_index_from_filename
2,WalkingAceleration5,/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/data/AnomalyArchive/162_UCR_Anomaly_WalkingAceleration5_2700_5920_5979.txt,6684,filename_anomaly_interval,5979,,705,59,anomaly_end_index_from_filename
3,DISTORTEDInternalBleeding9,/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/data/AnomalyArchive/036_UCR_Anomaly_DISTORTEDInternalBleeding9_4200_6599_6681.txt,7501,filename_anomaly_interval,6681,,820,82,anomaly_end_index_from_filename
4,InternalBleeding9,/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/data/AnomalyArchive/144_UCR_Anomaly_InternalBleeding9_4200_6599_6681.txt,7501,filename_anomaly_interval,6681,,820,82,anomaly_end_index_from_filename
5,mit14134longtermecg,/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/data/AnomalyArchive/229_UCR_Anomaly_mit14134longtermecg_16363_57960_57970.txt,59000,filename_anomaly_interval,57970,,1030,10,anomaly_end_index_from_filename
6,DISTORTEDInternalBleeding5,/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/data/AnomalyArchive/033_UCR_Anomaly_DISTORTEDInternalBleeding5_4000_6200_6370.txt,7415,filename_anomaly_interval,6370,,1045,170,anomaly_end_index_from_filename
7,InternalBleeding5,/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/data/AnomalyArchive/141_UCR_Anomaly_InternalBleeding5_4000_6200_6370.txt,7415,filename_anomaly_interval,6370,,1045,170,anomaly_end_index_from_filename
8,mit14134longtermecg,/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/data/AnomalyArchive/232_UCR_Anomaly_mit14134longtermecg_8763_57530_57790.txt,59000,filename_anomaly_interval,57790,,1210,260,anomaly_end_index_from_filename
9,DISTORTEDInternalBleeding8,/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/data/AnomalyArchive/035_UCR_Anomaly_DISTORTEDInternalBleeding8_2500_5865_5974.txt,7501,filename_anomaly_interval,5974,,1527,109,anomaly_end_index_from_filename


## SMD

,num_series,min_suffix,median_suffix,max_suffix,zero_suffix_count,short_suffix_count_le_100,no_anomaly_count
0,28,399,768.5,5962,0,0,0


...

,num_series,min_suffix,median_suffix,max_suffix,zero_suffix_count,short_suffix_count_le_100,no_anomaly_count
0,3,0,0.0,1387098,2,2,1


,series_id,source_file,total_length,annotation_type,anomaly_end_index,last_anomalous_index,post_anomaly_suffix_len,num_anomalous_points,notes
0,attack.csv,/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/data/SWaT/attack.csv,54621,attack_label_column,,54620.000000,0,54621,last_attack_label_in_csv
1,merged.csv,/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/data/SWaT/merged.csv,1441719,attack_label_column,,1441718.000000,0,54621,last_attack_label_in_csv
2,normal.csv,/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/data/SWaT/normal.csv,1387098,attack_label_column,,,1387098,0,no_anomaly_in_series


## IOPS

,num_series,min_suffix,median_suffix,max_suffix,zero_suffix_count,short_suffix_count_le_100,no_anomaly_count
0,29,27,2087.0,12392,0,1,0


,series_id,source_file,total_length,annotation_type,anomaly_end_index,last_anomalous_index,post_anomaly_suffix_len,num_anomalous_points,notes
0,KPI-54350a12-7a9d-3ca8-b81f-f886b9d156fd,/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/data/IOPS/KPI-54350a12-7a9d-3ca8-b81f-f886b9d156fd.test.out,7616,pointwise_label_column,,7588,27,108,last_nonzero_label_in_test_stream
1,KPI-55f8b8b8-b659-38df-b3df-e4a5a8a54bc9,/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/data/IOPS/KPI-55f8b8b8-b659-38df-b3df-e4a5a8a54bc9.test.out,149133,pointwise_label_column,,148993,139,6091,last_nonzero_label_in_test_stream
2,KPI-ba5f3328-9f3f-3ff5-a683-84437d16d554,/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/data/IOPS/KPI-ba5f3328-9f3f-3ff5-a683-84437d16d554.test.out,149132,pointwise_label_column,,148992,139,6777,last_nonzero_label_in_test_stream
3,KPI-a07ac296-de40-3a7c-8df3-91f642cc14d0,/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/data/IOPS/KPI-a07ac296-de40-3a7c-8df3-91f642cc14d0.test.out,111307,pointwise_label_column,,111073,233,2259,last_nonzero_label_in_test_stream
4,KPI-301c70d8-1630-35ac-8f96-bc1b6f4359ea,/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/data/IOPS/KPI-301c70d8-1630-35ac-8f96-bc1b6f4359ea.test.out,8784,pointwise_label_column,,8510,273,206,last_nonzero_label_in_test_stream
5,KPI-ab216663-dcc2-3a24-b1ee-2c3e550e06c9,/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/data/IOPS/KPI-ab216663-dcc2-3a24-b1ee-2c3e550e06c9.test.out,10780,pointwise_label_column,,10430,349,187,last_nonzero_label_in_test_stream
6,KPI-431a8542-c468-3988-a508-3afd06a218da,/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/data/IOPS/KPI-431a8542-c468-3988-a508-3afd06a218da.test.out,111566,pointwise_label_column,,111198,367,3286,last_nonzero_label_in_test_stream
7,KPI-da10a69f-d836-3baa-ad40-3e548ecf1fbd,/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/data/IOPS/KPI-da10a69f-d836-3baa-ad40-3e548ecf1fbd.test.out,107167,pointwise_label_column,,106536,630,8750,last_nonzero_label_in_test_stream
8,KPI-6d1114ae-be04-3c46-b5aa-be1a003a57cd,/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/data/IOPS/KPI-6d1114ae-be04-3c46-b5aa-be1a003a57cd.test.out,149122,pointwise_label_column,,147951,1170,761,last_nonzero_label_in_test_stream
9,KPI-adb2fde9-8589-3f5b-a410-5fe14386c7af,/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Khoá luận tốt nghiệp/bachelor-thesis-2026/data/IOPS/KPI-adb2fde9-8589-3f5b-a410-5fe14386c7af.test.out,149155,pointwise_label_column,,147904,1250,1692,last_nonzero_label_in_test_stream


In [8]:
# Nếu anh chỉ muốn soi các case đáng nghi nhất, chạy cell này.
focus_frames = {}

for dataset_name, df in tables.items():
    focus_frames[dataset_name] = df[
        (df["post_anomaly_suffix_len"] <= 100)
        | (df["notes"] == "no_anomaly_in_series")
        | (df["post_anomaly_suffix_len"] == 0)
    ].copy()

focus_frames


{'UCR': Empty DataFrame
 Columns: [series_id, source_file, total_length, annotation_type, anomaly_end_index, last_anomalous_index, post_anomaly_suffix_len, num_anomalous_points, notes]
 Index: [],
 'SMD': Empty DataFrame
 Columns: [series_id, source_file, total_length, annotation_type, anomaly_end_index, last_anomalous_index, post_anomaly_suffix_len, num_anomalous_points, notes]
 Index: [],
 'SWaT':     series_id  \
 0  attack.csv   
 1  merged.csv   
 2  normal.csv   
 
                                                                                                                source_file  \
 0  /Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Kh...   
 1  /Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Kh...   
 2  /Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/Kh...   
 
    total_leng